# Stage 2 Full Training: all categories and defect types

Trains one independent LoRA adapter per defect type across all three MVTec categories.

**12 adapters total:**
- bottle: broken_large, broken_small, contamination
- metal_nut: bent, color, flip, scratch
- leather: color, cut, fold, glue, poke

## Cell 1 - GPU check

In [ ]:
import torch, sys, os

print(f'Python  : {sys.version[:6]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU found. Enable GPU in notebook settings before continuing.')

## Cell 2 - Install dependencies
Exact versions confirmed working in the trial run.

In [ ]:
import subprocess, sys

packages = [
    "diffusers==0.29.2",
    "transformers==4.40.0",
    "accelerate==0.29.3",
    "peft==0.10.0",
    "wandb>=0.18.0",
    "safetensors",
    "Pillow",
    "tqdm",
    "pyyaml",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade",
    *packages
])

print("Installation complete. Verifying versions...\n")

import importlib
expected = {
    "diffusers":    "0.29",
    "transformers": "4.40",
    "accelerate":   "0.29",
    "peft":         "0.10",
}

all_ok = True
for pkg, prefix in expected.items():
    mod = importlib.import_module(pkg)
    ver = mod.__version__
    ok  = ver.startswith(prefix)
    print(f"  {'ok' if ok else 'X! WRONG'}  {pkg:<20} {ver}")
    if not ok:
        all_ok = False

import numpy as np
import wandb
print(f"  ok  numpy                {np.__version__}")
print(f"  ok  wandb                {wandb.__version__}")

if not all_ok:
    raise RuntimeError("Wrong package versions - restart kernel and re-run")
print("\nAll critical packages correct - safe to continue")

## Cell 3 - NumPy patch

In [ ]:
import numpy as np

if not hasattr(np, 'float_'):
    np.float_ = np.float64
if not hasattr(np, 'complex_'):
    np.complex_ = np.complex128

print(f'NumPy {np.__version__} - patch applied.')

## Cell 4 - Authenticate

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

secrets = UserSecretsClient()

hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)
print('HuggingFace : OK')

wandb_token = secrets.get_secret('WANDB_API_KEY')
wandb.login(key=wandb_token)
print('Wandb       : OK')

## Cell 5 - Clone repo and set working directory

In [ ]:
import os
from pathlib import Path

WORK     = '/kaggle/working'
REPO_URL = 'https://github.com/alecanc/Few-Shot-Personalization-of-a-Diffusion-Model-for-Industrial-Defect-Synthesis'
REPO_DIR = f'{WORK}/repo'

for d in ['checkpoints/stage2', 'splits', 'generated']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory : {os.getcwd()}')
print(f'Repo contents     : {sorted(os.listdir("."))}')

## Cell 6 - Mount MVTec dataset

If dataset is already extracted.  
Verify INPUT_DIR matches the path shown in the Kaggle Data sidebar.  
All three categories must be present for the full training run.

In [ ]:
from pathlib import Path

# ── Verify the path in the Kaggle sidebar > Data panel ──────────────────────
INPUT_DIR = Path('/kaggle/input/mvtec-anomaly-detection')
# ─────────────────────────────────────────────────────────────────────────────

MVTEC_ROOT = INPUT_DIR

CATEGORIES = {
    'bottle':    ['broken_large', 'broken_small', 'contamination'],
    'metal_nut': ['bent', 'color', 'flip', 'scratch'],
    'leather':   ['color', 'cut', 'fold', 'glue', 'poke'],
}

print('MVTec dataset verification:')
all_ok = True
for cat, dtypes in CATEGORIES.items():
    cat_dir = MVTEC_ROOT / cat
    if not cat_dir.exists():
        print(f'  X!  {cat} - NOT FOUND at {cat_dir}')
        all_ok = False
        continue
    n_clean = len(list((cat_dir / 'train/good').glob('*.png')))
    print(f'  ok  {cat} - {n_clean} clean train images')
    for dt in dtypes:
        dt_dir = cat_dir / 'test' / dt
        n = len(list(dt_dir.glob('*.png'))) if dt_dir.exists() else 0
        status = 'ok' if n > 0 else 'X! MISSING'
        print(f'       {status}  {dt:<20} {n} images')

if not all_ok:
    raise FileNotFoundError('Some categories are missing. Check INPUT_DIR.')
print('\nAll categories verified.')

## Cell 7 - Patch config.yaml and generate splits

Only overwrites runtime paths, all hyperparameters are trusted from the committed config.

In [ ]:
import yaml, json

CONFIG_PATH = f'{REPO_DIR}/defect-synthesis/config.yaml'

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['paths']['mvtec_root']  = str(MVTEC_ROOT)
cfg['paths']['output_root'] = WORK
cfg['paths']['checkpoints'] = f'{WORK}/checkpoints'
cfg['paths']['generated']   = f'{WORK}/generated'
cfg['paths']['splits_dir']  = f'{WORK}/splits'
cfg['paths']['logs']        = f'{WORK}/logs'

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('config.yaml - runtime paths applied.')
print(f"  model_id        : {cfg['model_id']}")
print(f"  mvtec_root      : {cfg['paths']['mvtec_root']}")
print(f"  max_train_steps : {cfg['stage2']['max_train_steps']}")
print(f"  n_per_type      : {cfg['splits']['stage2_n_per_type']}")
print()

# Generate splits for all three categories
!python data/splits.py --config {CONFIG_PATH}

# Verify all three split files and spot-check paths
print()
splits_dir = Path(f'{WORK}/splits')
for cat in ['bottle', 'metal_nut', 'leather']:
    split_file = splits_dir / f'{cat}_split.json'
    if not split_file.exists():
        print(f'  X!  {cat}_split.json NOT FOUND')
        continue
    with open(split_file) as f:
        split = json.load(f)
    # Spot-check first path exists on disk
    first_dtype = list(split['stage2'].keys())[0]
    first_path  = Path(split['stage2'][first_dtype][0])
    ok = 'ok' if first_path.exists() else 'X! PATH BROKEN'
    print(f'  {ok}  {cat} split - {sum(len(v) for v in split["stage2"].values())} defect images total')
    for dtype, imgs in split['stage2'].items():
        print(f'       {dtype:<20} {len(imgs)} train | {len(split["eval"].get(dtype, []))} eval')

---
## BOTTLE  3 defect types
Expected 30 minutes total

### Cell 8 - Train all bottle defect types

In [ ]:
CATEGORY     = 'bottle'
DEFECT_TYPES = ['broken_large', 'broken_small', 'contamination']

import shutil
from pathlib import Path

for defect_type in DEFECT_TYPES:
    print(f"\n{'='*60}")
    print(f'  Training: {CATEGORY} / {defect_type}')
    print(f"{'='*60}")
    !python -u train/train_stage2.py \
        --config {CONFIG_PATH} \
        --category {CATEGORY} \
        --defect_type {defect_type}

    source   = f'{WORK}/checkpoints/stage2/{CATEGORY}/{defect_type}'
    zip_base = f'{WORK}/stage2_{CATEGORY}_{defect_type}'
    shutil.make_archive(zip_base, 'zip', source)
    zip_path = Path(f'{zip_base}.zip')
    print(f'\n  > Zipped: {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)')
    print(f'     Download from the Output panel')

### Cell 9 - Verify bottle adapters

In [ ]:
from pathlib import Path

CATEGORY     = 'bottle'
DEFECT_TYPES = ['broken_large', 'broken_small', 'contamination']

print(f'Bottle adapter verification:')
all_ok = True
for dt in DEFECT_TYPES:
    final = Path(f'{WORK}/checkpoints/stage2/{CATEGORY}/{dt}/final')
    st    = next(final.glob('*.safetensors'), None) if final.exists() else None
    if st is None:
        print(f'  X!  {dt:<20} MISSING')
        all_ok = False
    else:
        size = st.stat().st_size / 1e6
        ok   = 1.0 < size < 50.0
        print(f'  {"ok" if ok else "X! WRONG SIZE"}  {dt:<20} {size:.1f} MB')
        if not ok:
            all_ok = False

print()
if all_ok:
    print('All bottle adapters OK - safe to continue.')
else:
    print('Some adapters failed - do NOT continue until fixed.')

### Cell 10 - Bottle visual sanity check

Generates 8 images per defect type.  
Saved as grids under `/kaggle/working/generated/`.

In [ ]:
import torch, yaml
from diffusers import StableDiffusionPipeline
from pathlib import Path
from PIL import Image as PILImage
from IPython.display import display
from peft import PeftModel

CATEGORY     = 'bottle'
DEFECT_TYPES = ['broken_large', 'broken_small', 'contamination']
N_IMAGES     = 8   # doubled for report
N_COLS       = 4   # 4 per row, 2 rows

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

model_id = cfg['model_id']
token_D  = cfg['token_D']
prompt   = f'a photo of a {token_D} defect on a surface'

# Load base pipeline once - reuse for all defect types
print(f'Loading base model: {model_id}')
base_pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
).to('cuda')
base_pipe.set_progress_bar_config(disable=True)

for defect_type in DEFECT_TYPES:
    adapter_dir = Path(f'{WORK}/checkpoints/stage2/{CATEGORY}/{defect_type}/final')
    if not adapter_dir.exists():
        print(f'  SKIP {defect_type} - adapter not found')
        continue

    print(f'\n{defect_type}:')

    # Load adapter on a fresh copy of the unet each time
    # to avoid cross-contamination between defect types
    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to('cuda')
    pipe.set_progress_bar_config(disable=True)
    pipe.unet = PeftModel.from_pretrained(pipe.unet, str(adapter_dir))
    pipe.vae.to(dtype=torch.float32)

    generator = torch.Generator('cuda').manual_seed(cfg['seed'])
    images    = pipe(
        prompt,
        num_images_per_prompt=N_IMAGES,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
    ).images

    # Build grid: N_COLS per row
    w, h    = images[0].size
    n_rows  = (N_IMAGES + N_COLS - 1) // N_COLS
    grid    = PILImage.new('RGB', (w * N_COLS, h * n_rows))
    for i, img in enumerate(images):
        grid.paste(img, ((i % N_COLS) * w, (i // N_COLS) * h))

    out_path = Path(f'{WORK}/generated/{CATEGORY}_{defect_type}.png')
    grid.save(out_path)
    print(f'  Saved: {out_path.name}')
    display(grid)

    del pipe
    torch.cuda.empty_cache()

print('\nBottle sanity checks complete.')

---
## METAL NUT  4 defect types
Expected 40 minutes total

### Cell 11 - Train all metal_nut defect types

In [ ]:
CATEGORY     = 'metal_nut'
DEFECT_TYPES = ['bent', 'color', 'flip', 'scratch']

import shutil
from pathlib import Path

for defect_type in DEFECT_TYPES:
    print(f"\n{'='*60}")
    print(f'  Training: {CATEGORY} / {defect_type}')
    print(f"{'='*60}")
    !python -u train/train_stage2.py \
        --config {CONFIG_PATH} \
        --category {CATEGORY} \
        --defect_type {defect_type}

    source   = f'{WORK}/checkpoints/stage2/{CATEGORY}/{defect_type}'
    zip_base = f'{WORK}/stage2_{CATEGORY}_{defect_type}'
    shutil.make_archive(zip_base, 'zip', source)
    zip_path = Path(f'{zip_base}.zip')
    print(f'\n  > Zipped: {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)')
    print(f'     Download from the Output panel')

### Cell 12 - Verify metal_nut adapters

In [ ]:
from pathlib import Path

CATEGORY     = 'metal_nut'
DEFECT_TYPES = ['bent', 'color', 'flip', 'scratch']

print(f'Metal nut adapter verification:')
all_ok = True
for dt in DEFECT_TYPES:
    final = Path(f'{WORK}/checkpoints/stage2/{CATEGORY}/{dt}/final')
    st    = next(final.glob('*.safetensors'), None) if final.exists() else None
    if st is None:
        print(f'  X!  {dt:<20} MISSING')
        all_ok = False
    else:
        size = st.stat().st_size / 1e6
        ok   = 1.0 < size < 50.0
        print(f'  {"ok" if ok else "X! WRONG SIZE"}  {dt:<20} {size:.1f} MB')
        if not ok:
            all_ok = False

print()
if all_ok:
    print('All metal_nut adapters OK - safe to continue.')
else:
    print('Some adapters failed - do NOT continue until fixed.')

### Cell 13 - Metal nut visual sanity check

In [ ]:
import torch, yaml
from diffusers import StableDiffusionPipeline
from pathlib import Path
from PIL import Image as PILImage
from IPython.display import display
from peft import PeftModel

CATEGORY     = 'metal_nut'
DEFECT_TYPES = ['bent', 'color', 'flip', 'scratch']
N_IMAGES     = 8
N_COLS       = 4

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

model_id = cfg['model_id']
token_D  = cfg['token_D']
prompt   = f'a photo of a {token_D} defect on a surface'

for defect_type in DEFECT_TYPES:
    adapter_dir = Path(f'{WORK}/checkpoints/stage2/{CATEGORY}/{defect_type}/final')
    if not adapter_dir.exists():
        print(f'  SKIP {defect_type} - adapter not found')
        continue

    print(f'\n{defect_type}:')

    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to('cuda')
    pipe.set_progress_bar_config(disable=True)
    pipe.unet = PeftModel.from_pretrained(pipe.unet, str(adapter_dir))
    pipe.vae.to(dtype=torch.float32)

    generator = torch.Generator('cuda').manual_seed(cfg['seed'])
    images    = pipe(
        prompt,
        num_images_per_prompt=N_IMAGES,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
    ).images

    w, h   = images[0].size
    n_rows = (N_IMAGES + N_COLS - 1) // N_COLS
    grid   = PILImage.new('RGB', (w * N_COLS, h * n_rows))
    for i, img in enumerate(images):
        grid.paste(img, ((i % N_COLS) * w, (i // N_COLS) * h))

    out_path = Path(f'{WORK}/generated/{CATEGORY}_{defect_type}.png')
    grid.save(out_path)
    print(f'  Saved: {out_path.name}')
    display(grid)

    del pipe
    torch.cuda.empty_cache()

print('\nMetal nut sanity checks complete.')

---
## LEATHER - 5 defect types
Expected 50 minutes total

### Cell 14 - Train all leather defect types

In [ ]:
CATEGORY     = 'leather'
DEFECT_TYPES = ['color', 'cut', 'fold', 'glue', 'poke']

import shutil
from pathlib import Path

for defect_type in DEFECT_TYPES:
    print(f"\n{'='*60}")
    print(f'  Training: {CATEGORY} / {defect_type}')
    print(f"{'='*60}")
    !python -u train/train_stage2.py \
        --config {CONFIG_PATH} \
        --category {CATEGORY} \
        --defect_type {defect_type}

    source   = f'{WORK}/checkpoints/stage2/{CATEGORY}/{defect_type}'
    zip_base = f'{WORK}/stage2_{CATEGORY}_{defect_type}'
    shutil.make_archive(zip_base, 'zip', source)
    zip_path = Path(f'{zip_base}.zip')
    print(f'\n  > Zipped: {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)')
    print(f'     Download from the Output panel')

### Cell 15 - Verify leather adapters

In [ ]:
from pathlib import Path

CATEGORY     = 'leather'
DEFECT_TYPES = ['color', 'cut', 'fold', 'glue', 'poke']

print(f'Leather adapter verification:')
all_ok = True
for dt in DEFECT_TYPES:
    final = Path(f'{WORK}/checkpoints/stage2/{CATEGORY}/{dt}/final')
    st    = next(final.glob('*.safetensors'), None) if final.exists() else None
    if st is None:
        print(f'  X!  {dt:<20} MISSING')
        all_ok = False
    else:
        size = st.stat().st_size / 1e6
        ok   = 1.0 < size < 50.0
        print(f'  {"ok" if ok else "X! WRONG SIZE"}  {dt:<20} {size:.1f} MB')
        if not ok:
            all_ok = False

print()
if all_ok:
    print('All leather adapters OK - safe to continue.')
else:
    print('Some adapters failed - do NOT continue until fixed.')

### Cell 16 - Leather visual sanity check

In [ ]:
import torch, yaml
from diffusers import StableDiffusionPipeline
from pathlib import Path
from PIL import Image as PILImage
from IPython.display import display
from peft import PeftModel

CATEGORY     = 'leather'
DEFECT_TYPES = ['color', 'cut', 'fold', 'glue', 'poke']
N_IMAGES     = 8
N_COLS       = 4

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

model_id = cfg['model_id']
token_D  = cfg['token_D']
prompt   = f'a photo of a {token_D} defect on a surface'

for defect_type in DEFECT_TYPES:
    adapter_dir = Path(f'{WORK}/checkpoints/stage2/{CATEGORY}/{defect_type}/final')
    if not adapter_dir.exists():
        print(f'  SKIP {defect_type} - adapter not found')
        continue

    print(f'\n{defect_type}:')

    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to('cuda')
    pipe.set_progress_bar_config(disable=True)
    pipe.unet = PeftModel.from_pretrained(pipe.unet, str(adapter_dir))
    pipe.vae.to(dtype=torch.float32)

    generator = torch.Generator('cuda').manual_seed(cfg['seed'])
    images    = pipe(
        prompt,
        num_images_per_prompt=N_IMAGES,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
    ).images

    w, h   = images[0].size
    n_rows = (N_IMAGES + N_COLS - 1) // N_COLS
    grid   = PILImage.new('RGB', (w * N_COLS, h * n_rows))
    for i, img in enumerate(images):
        grid.paste(img, ((i % N_COLS) * w, (i // N_COLS) * h))

    out_path = Path(f'{WORK}/generated/{CATEGORY}_{defect_type}.png')
    grid.save(out_path)
    print(f'  Saved: {out_path.name}')
    display(grid)

    del pipe
    torch.cuda.empty_cache()

print('\nLeather sanity checks complete.')

---
## Cell 17 - Final verification: all 12 adapters

In [ ]:
from pathlib import Path

CATEGORIES = {
    'bottle':    ['broken_large', 'broken_small', 'contamination'],
    'metal_nut': ['bent', 'color', 'flip', 'scratch'],
    'leather':   ['color', 'cut', 'fold', 'glue', 'poke'],
}

print('Final adapter inventory:')
print()
total_ok = 0
total    = sum(len(v) for v in CATEGORIES.values())

for cat, dtypes in CATEGORIES.items():
    print(f'  {cat}:')
    for dt in dtypes:
        final = Path(f'{WORK}/checkpoints/stage2/{cat}/{dt}/final')
        st    = next(final.glob('*.safetensors'), None) if final.exists() else None
        if st is None:
            print(f'    X!  {dt:<20} MISSING')
        else:
            size = st.stat().st_size / 1e6
            ok   = 1.0 < size < 50.0
            print(f'    {"ok" if ok else "X!"}  {dt:<20} {size:.1f} MB')
            if ok:
                total_ok += 1
    print()

print(f'Result: {total_ok}/{total} adapters ready.')
if total_ok == total:
    print('All adapters present - safe to zip and save.')
else:
    print(f'{total - total_ok} adapter(s) missing - check training output above.')

## Cell 18 - Zip all adapters and generated images

Creates two zip files:
- `stage2_all_adapters.zip` — all 12 LoRA adapter directories (~60 MB)
- `stage2_all_generated.zip` — all 12 sanity check image grids

Download from the Output panel in the right sidebar, or click Save Version

In [ ]:
import shutil
from pathlib import Path

# Zip all generated grids for the report
generated_source = f'{WORK}/generated'
generated_zip    = f'{WORK}/stage2_all_generated'
shutil.make_archive(generated_zip, 'zip', generated_source)
generated_path = Path(f'{generated_zip}.zip')
print(f'Generated zip: {generated_path.name}  ({generated_path.stat().st_size/1e6:.1f} MB)')
print()
print('Download from the Output panel > right sidebar,')
print('click Save Version to commit as notebook output')